# 36. Neural Networks: Generative Adversarial Networks (GANs)

## Algorithm Category
**Type**: Neural Networks - Generative Models  
**Complexity**: Very High  
**Use Case**: Image generation, data augmentation, unsupervised learning

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand GAN architecture and adversarial training
- Implement a simple GAN from scratch
- Understand generator and discriminator networks
- Train GANs and visualize generated samples
- Understand common GAN challenges and solutions

## Historical Context

GANs were introduced by Goodfellow et al. in 2014:
- Goodfellow, I., et al. (2014): "Generative Adversarial Nets"
- Revolutionary approach to generative modeling
- Foundation for modern image generation

**Key Papers/References:**
- Goodfellow, I., et al. (2014). "Generative Adversarial Nets"
- Radford, A., et al. (2015). "Unsupervised representation learning with deep convolutional generative adversarial networks"

## When to Use GANs

GANs are appropriate when:
- Need to generate realistic data
- Image generation and synthesis
- Data augmentation
- Unsupervised learning
- Style transfer
- When you have large datasets

## Theory & Mechanics

### Mathematical Foundation

**GAN Objective (Minimax Game):**
$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] + \mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]$$

Where:
- $G$: Generator network
- $D$: Discriminator network
- $p_{data}$: Real data distribution
- $p_z$: Noise distribution (latent space)

**Generator Loss:**
$$\mathcal{L}_G = -\mathbb{E}_{z \sim p_z(z)}[\log D(G(z))]$$

**Discriminator Loss:**
$$\mathcal{L}_D = -\mathbb{E}_{x \sim p_{data}(x)}[\log D(x)] - \mathbb{E}_{z \sim p_z(z)}[\log(1-D(G(z)))]$$

### Key Components

1. **Generator (G)**
   - Takes random noise as input
   - Generates fake data
   - Tries to fool discriminator
   - Learns data distribution

2. **Discriminator (D)**
   - Classifies real vs fake
   - Tries to distinguish real from generated
   - Provides feedback to generator

3. **Adversarial Training**
   - Two networks compete
   - Generator improves to fool discriminator
   - Discriminator improves to detect fakes
   - Equilibrium when generator produces realistic data

### How It Works

1. **Initialize**: Random generator and discriminator
2. **Train Discriminator**: On real and fake data
3. **Train Generator**: To fool discriminator
4. **Alternate**: Between training D and G
5. **Converge**: When generator produces realistic data

### Key Hyperparameters

- **latent_dim**: Dimension of noise vector
- **learning_rate**: Step size (often different for G and D)
- **batch_size**: Number of samples per batch
- **n_critic**: Number of D updates per G update
- **beta1, beta2**: Adam optimizer parameters

### Advantages

- Can generate realistic data
- Unsupervised learning
- No explicit likelihood needed
- Produces diverse samples
- State-of-the-art image generation

### Limitations

- Training instability
- Mode collapse (limited diversity)
- Hard to evaluate
- Requires careful tuning
- Computationally expensive


## Implementation

Let's implement a simple GAN for generating 2D data.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Libraries imported successfully!")


In [ ]:
# Generate simple 2D data (circle)
def generate_circle_data(n_samples=1000):
    """Generate 2D circle data"""
    angles = np.random.uniform(0, 2*np.pi, n_samples)
    radius = 1.0
    x = radius * np.cos(angles) + np.random.normal(0, 0.1, n_samples)
    y = radius * np.sin(angles) + np.random.normal(0, 0.1, n_samples)
    return np.column_stack([x, y])

# Generate real data
real_data = generate_circle_data(n_samples=1000)
real_data = (real_data - real_data.mean(axis=0)) / real_data.std(axis=0)  # Normalize

print(f"Real data shape: {real_data.shape}")

# Visualize real data
plt.figure(figsize=(6, 6))
plt.scatter(real_data[:, 0], real_data[:, 1], alpha=0.5, s=20)
plt.title('Real Data (Circle)')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()


In [ ]:
# Generator Network
class Generator(nn.Module):
    def __init__(self, latent_dim=2, output_dim=2, hidden_dim=64):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z):
        return self.net(z)

# Discriminator Network
class Discriminator(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()  # Probability of being real
        )
    
    def forward(self, x):
        return self.net(x)

print("Generator and Discriminator models defined!")


In [ ]:
# Initialize models
latent_dim = 2
generator = Generator(latent_dim=latent_dim, output_dim=2, hidden_dim=64).to(device)
discriminator = Discriminator(input_dim=2, hidden_dim=64).to(device)

# Loss and optimizers
criterion = nn.BCELoss()
lr = 0.0002
beta1 = 0.5
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

# Prepare real data
real_data_tensor = torch.FloatTensor(real_data).to(device)
real_labels = torch.ones(len(real_data), 1).to(device)
fake_labels = torch.zeros(len(real_data), 1).to(device)

print(f"Generator parameters: {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")


## Training

Let's train the GAN.


In [ ]:
# Training loop
num_epochs = 200
batch_size = 64
n_critic = 1  # Number of D updates per G update

G_losses = []
D_losses = []

for epoch in range(num_epochs):
    # Train Discriminator
    for _ in range(n_critic):
        # Real data
        idx = torch.randint(0, len(real_data), (batch_size,))
        real_batch = real_data_tensor[idx]
        real_label = real_labels[:batch_size]
        
        # Fake data
        noise = torch.randn(batch_size, latent_dim).to(device)
        fake_batch = generator(noise)
        fake_label = fake_labels[:batch_size]
        
        # Train D on real
        optimizer_D.zero_grad()
        D_real = discriminator(real_batch)
        loss_D_real = criterion(D_real, real_label)
        
        # Train D on fake
        D_fake = discriminator(fake_batch.detach())
        loss_D_fake = criterion(D_fake, fake_label)
        
        # Total D loss
        loss_D = (loss_D_real + loss_D_fake) / 2
        loss_D.backward()
        optimizer_D.step()
    
    # Train Generator
    optimizer_G.zero_grad()
    noise = torch.randn(batch_size, latent_dim).to(device)
    fake_batch = generator(noise)
    D_fake = discriminator(fake_batch)
    loss_G = criterion(D_fake, real_label)  # Try to fool D
    loss_G.backward()
    optimizer_G.step()
    
    # Save losses
    G_losses.append(loss_G.item())
    D_losses.append(loss_D.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: D_loss = {loss_D.item():.4f}, G_loss = {loss_G.item():.4f}")

print("\nTraining complete!")


## Visualization

Let's visualize the training progress and generated samples.


In [ ]:
# Plot training losses
plt.figure(figsize=(10, 6))
plt.plot(G_losses, label='Generator Loss', alpha=0.7)
plt.plot(D_losses, label='Discriminator Loss', alpha=0.7)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GAN Training Losses')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Generate samples
generator.eval()
with torch.no_grad():
    noise = torch.randn(1000, latent_dim).to(device)
    generated_data = generator(noise).cpu().numpy()

# Visualize real vs generated
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].scatter(real_data[:, 0], real_data[:, 1], alpha=0.5, s=20, c='blue')
axes[0].set_title('Real Data')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

axes[1].scatter(generated_data[:, 0], generated_data[:, 1], alpha=0.5, s=20, c='red')
axes[1].set_title('Generated Data')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()


## Validation & Testing

Let's validate the GAN performance.


In [ ]:
# Check discriminator accuracy
discriminator.eval()
with torch.no_grad():
    # Real data
    D_real_pred = discriminator(real_data_tensor[:100])
    real_acc = (D_real_pred > 0.5).float().mean().item()
    
    # Generated data
    noise = torch.randn(100, latent_dim).to(device)
    fake_data = generator(noise)
    D_fake_pred = discriminator(fake_data)
    fake_acc = (D_fake_pred < 0.5).float().mean().item()

print("Discriminator Performance:")
print(f"  Real data accuracy: {real_acc:.3f}")
print(f"  Fake data accuracy: {fake_acc:.3f}")

# Assertions
assert len(G_losses) == num_epochs, "Should have trained for all epochs"
print("\n✓ Validation checks passed")

print("\nNote: GANs are notoriously difficult to train.")
print("Key challenges include mode collapse, training instability,")
print("and difficulty in evaluation.")


## Summary & Key Takeaways

### Key Concepts Learned

1. **GAN Architecture**
   - Two competing networks: Generator and Discriminator
   - Adversarial training process
   - Minimax game formulation
   - No explicit likelihood needed

2. **Generator (G)**
   - Maps noise to data space
   - Learns to generate realistic samples
   - Tries to fool discriminator
   - Improves through adversarial feedback

3. **Discriminator (D)**
   - Classifies real vs fake
   - Provides training signal to generator
   - Improves to detect fakes
   - Should converge to 0.5 when G is perfect

4. **Training Challenges**
   - **Mode collapse**: Generator produces limited diversity
   - **Instability**: Hard to balance G and D
   - **Evaluation**: No clear metric for quality
   - **Convergence**: May not converge to Nash equilibrium

### When to Use GANs

✅ **Good for:**
- Image generation
- Data augmentation
- Unsupervised learning
- Style transfer
- When you need realistic samples
- Large datasets available

❌ **Not ideal for:**
- Small datasets
- When interpretability is needed
- Real-time applications
- When stability is critical
- When explicit likelihood is needed

### Next Steps

- Explore **DCGAN** (Deep Convolutional GAN) for images
- Try **WGAN** (Wasserstein GAN) for stability
- Use **Conditional GANs** for controlled generation
- Apply **Progressive GANs** for high-resolution images
- Experiment with **StyleGAN** for advanced image synthesis
